# Orchestrator-Worker with Interceptor Pattern

---

This notebook demonstrates a **clean separation of concerns**:

| Layer | Concern | Mechanism |
|---|---|---|
| **Chains / Tools** | Per-capability ops (retry, fallback, cache, rate-limit) | Decorate the chain/tool directly |
| **Node functions** | Business logic only (`state → partial update`) | Plain Python functions |
| **Graph (InstrumentedGraph)** | Universal cross-cutting (logging, metrics, tracing) | Subclass with interceptor hooks |
| **Invocation** | Run-level config (callbacks, run tags, recursion limit) | `RunnableConfig` at `.invoke()` |

### Key insight

Retry, fallback, and caching belong on the **chain or tool** — the thing that actually
talks to an LLM or external API. Nodes are just state routing; they don't need ops policy.

Universal observability (logging, metrics) belongs on the **graph** via a subclass that
wraps every node automatically — no opt-in required.

---
# Setup
---

In [ ]:
from __future__ import annotations

import functools
import logging
import operator
import threading
import time
from abc import ABC
from collections import defaultdict
from pathlib import Path
from pprint import pprint
from typing import Annotated, Any, List, TypedDict

from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field

In [ ]:
env_path = Path.home() / "Source" / "SECRETS" / ".env"
load_dotenv(env_path)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

---
# Interceptor Framework

The `GraphInterceptor` ABC defines hooks for universal cross-cutting concerns.

`InstrumentedGraph` subclasses `StateGraph` and wraps every node automatically.

---

In [ ]:
class GraphInterceptor(ABC):
    """Base class for graph interceptors.

    Subclass and override only the hooks you need.
    All hooks have no-op defaults so you can be selective.
    """

    def on_node_start(self, node_name: str, state: dict) -> None:
        pass

    def on_node_end(self, node_name: str, state: dict, result: dict) -> None:
        pass

    def on_node_error(self, node_name: str, state: dict, error: Exception) -> None:
        pass

In [ ]:
class InstrumentedGraph(StateGraph):
    """StateGraph subclass that wraps every node with interceptor hooks.

    Usage:
        graph = InstrumentedGraph(MyState, interceptors=[LoggingInterceptor()])
        graph.add_node("my_node", my_plain_function)
    """

    def __init__(self, *args, interceptors: list[GraphInterceptor] | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._interceptors: list[GraphInterceptor] = interceptors or []

    def add_node(self, node: str, action, **kwargs):
        """Override: wrap the action with interceptor hooks, then delegate."""
        wrapped = self._wrap_with_interceptors(action, node)
        super().add_node(node, wrapped, **kwargs)

    def _wrap_with_interceptors(self, func, node_name: str):
        """Wrap a node function so all interceptors fire around it."""
        interceptors = self._interceptors

        @functools.wraps(func)
        def wrapper(state):
            for ic in interceptors:
                try:
                    ic.on_node_start(node_name, state)
                except Exception:
                    pass  # interceptors must not break execution

            try:
                result = func(state)
            except Exception as exc:
                for ic in interceptors:
                    try:
                        ic.on_node_error(node_name, state, exc)
                    except Exception:
                        pass
                raise

            for ic in interceptors:
                try:
                    ic.on_node_end(node_name, state, result)
                except Exception:
                    pass

            return result

        return wrapper

---
# Concrete Interceptors
---

In [ ]:
class LoggingInterceptor(GraphInterceptor):
    """Logs node start / end / error events."""

    def __init__(self, logger: logging.Logger | None = None):
        self.logger = logger or logging.getLogger("langgraph.interceptor")

    def on_node_start(self, node_name: str, state: dict) -> None:
        self.logger.info("[%s] starting", node_name)

    def on_node_end(self, node_name: str, state: dict, result: dict) -> None:
        self.logger.info("[%s] completed", node_name)

    def on_node_error(self, node_name: str, state: dict, error: Exception) -> None:
        self.logger.error("[%s] failed: %s", node_name, error)

In [ ]:
class MetricsInterceptor(GraphInterceptor):
    """Tracks per-node timing and error counts.

    Thread-safe: uses (node_name, thread_id) as the timing key so
    parallel Send() fan-outs don't clobber each other.
    """

    def __init__(self):
        self._lock = threading.Lock()
        self._start_times: dict[tuple[str, int], float] = {}
        self.node_timings: dict[str, list[float]] = defaultdict(list)
        self.node_counts: dict[str, int] = defaultdict(int)
        self.node_errors: dict[str, int] = defaultdict(int)

    def _key(self, node_name: str) -> tuple[str, int]:
        return (node_name, threading.get_ident())

    def on_node_start(self, node_name: str, state: dict) -> None:
        self._start_times[self._key(node_name)] = time.perf_counter()

    def on_node_end(self, node_name: str, state: dict, result: dict) -> None:
        key = self._key(node_name)
        elapsed = time.perf_counter() - self._start_times.pop(key, time.perf_counter())
        with self._lock:
            self.node_timings[node_name].append(elapsed)
            self.node_counts[node_name] += 1

    def on_node_error(self, node_name: str, state: dict, error: Exception) -> None:
        self._start_times.pop(self._key(node_name), None)
        with self._lock:
            self.node_errors[node_name] += 1

    def get_stats(self) -> dict[str, dict[str, Any]]:
        with self._lock:
            stats = {}
            for name, timings in self.node_timings.items():
                stats[name] = {
                    "count": self.node_counts[name],
                    "errors": self.node_errors[name],
                    "avg_s": sum(timings) / len(timings) if timings else 0,
                    "min_s": min(timings) if timings else 0,
                    "max_s": max(timings) if timings else 0,
                }
            return stats

---
# Domain Models & State
---

In [ ]:
class Dish(BaseModel):
    name: str = Field(
        description="Name of the dish (for example, Spaghetti Bolognese, Chicken Curry)."
    )
    ingredients: List[str] = Field(
        description="List of ingredients needed for this dish, separated by commas."
    )
    location: str = Field(
        description="The cuisine or cultural origin of the dish (for example, Italian, Indian, Mexican)."
    )


class Dishes(BaseModel):
    sections: List[Dish] = Field(
        description="A list of grocery sections, one for each dish, with ingredients."
    )

In [ ]:
class State(TypedDict):
    meals: str
    sections: List[Dish]
    completed_menu: Annotated[List[str], operator.add]
    final_meal_guide: str


class WorkerState(TypedDict):
    section: Dish
    completed_menu: Annotated[list, operator.add]

---
# Chains / Tools — Per-Capability Ops Policy

Retry, fallback, caching, and rate-limiting belong **here**, on the thing
that actually talks to the LLM or external API.

The node function just calls the chain — it doesn't know or care about retry policy.

---

In [ ]:
dish_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an assistant that generates a structured grocery list.\n\n"
        "The user wants to prepare the following meals: {meals}\n\n"
        "For each meal, return a section with:\n"
        "- the name of the dish\n"
        "- a comma-separated list of ingredients needed for that dish.\n"
        "- the cuisine or cultural origin of the food"
    )
])

chef_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a world-class chef from {location}.\n\n"
        "Please introduce yourself briefly and present a detailed walkthrough for preparing the dish: {name}.\n"
        "Your response should include:\n"
        "- Start with hello with your  name and culinary background\n"
        "- A clear list of preparation steps\n"
        "- A full explanation of the cooking process\n\n"
        "Use the following ingredients: {ingredients}."
    )
])

In [ ]:
# Ops policy lives on the chain, not on the node.
# .with_retry() wraps only the LLM call — if it 429s or 5xxs, it retries
# the chain invocation, not the entire node's state logic.

planner_pipe = (
    dish_prompt
    | llm.with_structured_output(Dishes)
).with_retry(stop_after_attempt=3)

chef_pipe = (
    chef_prompt
    | llm
).with_retry(stop_after_attempt=3)

---
# Node Functions — Pure Business Logic

Each node is a plain function: `state → partial state update`.

- No retry, timeout, or logging inside the function.
- No awareness of which graph it's in.
- Testable with a plain dict input and a plain dict assertion.

---

In [ ]:
def orchestrator(state: State) -> dict:
    """Generate a structured dish list from the given meals."""
    dish_descriptions = planner_pipe.invoke({"meals": state["meals"]})
    return {"sections": dish_descriptions.sections}


def chef_worker(state: WorkerState) -> dict:
    """Generate cooking instructions for one meal section."""
    meal_plan = chef_pipe.invoke({
        "name": state["section"].name,
        "location": state["section"].location,
        "ingredients": state["section"].ingredients,
    })
    return {"completed_menu": [meal_plan.content]}


def synthesizer(state: State) -> dict:
    """Synthesize full report from completed sections."""
    completed_menu = "\n\n---\n\n".join(state["completed_menu"])
    return {"final_meal_guide": completed_menu}


def assign_workers(state: State):
    """Fan-out: assign a worker to each section in the plan."""
    return [
        Send("chef_worker", {"section": s, "completed_menu": []})
        for s in state["sections"]
    ]

---
# Graph Assembly

Use `InstrumentedGraph` instead of `StateGraph`. Every node gets interceptor
hooks automatically. No `wrap_node()` needed — the subclass handles it.

---

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

log_interceptor = LoggingInterceptor()
metrics_interceptor = MetricsInterceptor()

In [ ]:
builder = InstrumentedGraph(
    State,
    interceptors=[log_interceptor, metrics_interceptor],
)

builder.add_node("orchestrator", orchestrator)
builder.add_node("chef_worker", chef_worker)
builder.add_node("synthesizer", synthesizer)

builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", assign_workers, ["chef_worker"])
builder.add_edge("chef_worker", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

---
# Invocation — Run-Level Config

Run-level tags, metadata, and callbacks go here.

---

In [ ]:
def make_run_config(
    *,
    env: str = "dev",
    graph_name: str = "orchestrator_worker",
    graph_version: str = "0.2.0",
    user_id: str | None = None,
    request_id: str | None = None,
    callbacks: list | None = None,
) -> RunnableConfig:
    """Build a run-level config dict for graph.invoke()."""
    tags = [f"env:{env}", f"graph:{graph_name}", f"version:{graph_version}"]
    meta: dict[str, Any] = {
        "env": env,
        "graph_name": graph_name,
        "graph_version": graph_version,
    }
    if user_id:
        meta["user_id"] = user_id
    if request_id:
        meta["request_id"] = request_id

    config: RunnableConfig = {"tags": tags, "metadata": meta}
    if callbacks:
        config["callbacks"] = callbacks
    return config

In [ ]:
run_config = make_run_config(env="dev", user_id="chris")

result = graph.invoke(
    {"meals": "Steak and eggs, tacos, and chili"},
    config=run_config,
)

pprint(result["final_meal_guide"][:2000])

In [ ]:
print("\n=== Node Metrics ===")
pprint(metrics_interceptor.get_stats())

---
# Design Summary

### Where each concern lives

| Concern | Mechanism | Example |
|---|---|---|
| **Retry / fallback** | `.with_retry()` / `.with_fallbacks()` on the chain | `chef_pipe.with_retry(stop_after_attempt=3)` |
| **Caching** | Decorator or `.with_cache()` on the tool/chain | `@with_cache(ttl=3600)` |
| **Rate limiting** | LLM client config or decorator on the tool | `ChatOpenAI(max_retries=..., timeout=...)` |
| **Logging** | `LoggingInterceptor` on `InstrumentedGraph` | Fires for every node automatically |
| **Metrics** | `MetricsInterceptor` on `InstrumentedGraph` | Fires for every node automatically |
| **Tracing (OTel)** | `OTelInterceptor` on `InstrumentedGraph` | Fires for every node automatically |
| **Run tags / metadata** | `make_run_config()` at invocation | `graph.invoke(input, config=run_config)` |

### Why this works

- **Nodes stay pure** — testable with a plain dict, no ops awareness
- **Retry targets the right thing** — the LLM call, not the state routing
- **Interceptors can't be bypassed** — subclass enforces wrapping on every `add_node()`
- **Interceptors are defensive** — each hook is try/excepted so a broken interceptor can't crash the graph
- **Thread-safe metrics** — keyed by `(node_name, thread_id)` for parallel `Send()` fan-outs
- **`wrap_node()` is eliminated** — no free function to forget to call

---